## Alpha Earth Random Extraction

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import time
import datetime
import ee
import geemap.foliumap as geemap  
import datetime

In [2]:
# Authenticate ee. You first need to set up a free Google Cloud account
ee.Authenticate()

True

In [3]:
# Initialise Earth Engine
ee.Initialize()  

In [ ]:
# plot problem location

i=10617                     
i=10642               
i=10625

lat = gdf.iloc[i,1]
lon = gdf.iloc[i,2]
end_year = str(gdf.iloc[i,4])
start_date = end_year + '-01-01'
end_date = end_year + '-12-31'

print(lon,lat,end_year,start_date,end_date)

point = ee.Geometry.Point(lon, lat)    
box = ee.Geometry.BBox((lon-0.001), (lat-0.001), (lon+0.001), (lat+0.001))             
box_fc =  ee.FeatureCollection(box).style(**{'color':'blue','fillColor': '#00000000'}) 

image = (
    ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')     
    .filterDate(start_date, end_date)           
    .filterBounds(box)
)

sample = image.median().sample(point, scale=10)
sample_dict = sample.first().toDictionary().getInfo()
sample_np = np.array(list(sample_dict.values()), dtype=float)

Map = geemap.Map(height='400', width='40%')
Map.centerObject(point,zoom=14)

vis = {'min':-0.3, 'max':0.3, 'bands': ['A01','A02','A03']}     
Map.addLayer(image,vis,'image') 
Map.addLayer(box_fc, {}, "box_fc")

# Add centre point
Map.addLayer(point,{"color": "red"}, "point")    

Map

## Read random sample locations

In [4]:
#####
#region = 'biomeAll_southern_africa'
region = 'biomeAll_western_usa'
year = '2025'
region, year

('biomeAll_western_usa', '2025')

In [5]:
# read geodataframe from file
filename = '../Inference_Points/' + region + '_random_points.parquet'
print(filename)

../Inference_Points/biomeAll_western_usa_random_points.parquet


In [6]:
gdf = gpd.read_parquet(filename)  
len(gdf)

37789

In [7]:
# add lon and lat columns
gdf['lon'] = gdf.geometry.x
gdf['lat'] = gdf.geometry.y

In [8]:
gdf.head(2)

,geometry,lon,lat
0,POINT (-116.78916 34.38031),-116.789158,34.380307
1,POINT (-96.88028 28.41738),-96.880280,28.417383


## Extract Inference Data from AlphaEarth classifier

In [9]:
print(datetime.datetime.now())
time_start = time.time()

print('no of inference samples =',len(gdf))

start_sample = 0
end_sample = len(gdf)

start_date = year + '-01-01'
end_date = year + '-12-31'

print('year = ',start_date, 'to', end_date)

print('from',start_sample,'to',end_sample)

sample_all = np.zeros((0,64))

for i in range(start_sample,end_sample):    
    lon = gdf.iloc[i,1]
    lat = gdf.iloc[i,2]
    
    point = ee.Geometry.Point(lon, lat)   

    point = ee.Geometry.Point(lon, lat)    
    box = ee.Geometry.BBox((lon-0.001), (lat-0.001), (lon+0.001), (lat+0.001))             
    box_fc =  ee.FeatureCollection(box).style(**{'color':'blue','fillColor': '#00000000'}) 

    image = (
        ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')     
        .filterDate(start_date, end_date)           
        .filterBounds(box)                       # using box rather than points helps if point is directly on image boundary
    )

    sample = image.median().sample(point, scale=10)
    sample_dict = sample.first().toDictionary().getInfo()
    sample_np = np.array(list(sample_dict.values()), dtype=float)

    sample_np = np.expand_dims(sample_np,0)  
    sample_all = np.concatenate((sample_all,sample_np),axis=0)

    if i % 10 == 0: 
        print('sample = ', i)

print('\nsample_all: ',sample_all.shape)
print(region,year)

time_end = time.time()
print('\n\nfinal run time=',(time_end-time_start)/60," minutes")

2026-06-04 05:40:09.156780
no of inference samples = 37789
year =  2025-01-01 to 2025-12-31
from 0 to 37789
sample =  0
sample =  10
sample =  20
sample =  30
sample =  40
sample =  50
sample =  60
sample =  70
sample =  80
sample =  90
sample =  100
sample =  110
sample =  120
sample =  130
sample =  140
sample =  150
sample =  160
sample =  170
sample =  180
sample =  190
sample =  200
sample =  210
sample =  220
sample =  230
sample =  240
sample =  250
sample =  260
sample =  270
sample =  280
sample =  290
sample =  300
sample =  310
sample =  320
sample =  330
sample =  340
sample =  350
sample =  360
sample =  370
sample =  380
sample =  390
sample =  400
sample =  410
sample =  420
sample =  430
sample =  440
sample =  450
sample =  460
sample =  470
sample =  480
sample =  490
sample =  500
sample =  510
sample =  520
sample =  530
sample =  540
sample =  550
sample =  560
sample =  570
sample =  580
sample =  590
sample =  600
sample =  610
sample =  620
sample =  630
sample 

In [10]:
filename = '../Inference_Points/' + region + '_random_embeddings_' + year + '.npy'
filename

'../Inference_Points/biomeAll_western_usa_random_embeddings_2025.npy'

In [11]:
# save pixel data
np.save(filename, sample_all)

## Load Random Data 

In [ ]:
# load pixel data
pixel_loaded = np.load(filename)
pixel_loaded.shape